In [84]:
# Basic imports
import re
import random
import numpy as np
import pandas as pd

# Sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

# Stemming
from nltk.stem import PorterStemmer

# Seed for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)


In [ ]:
# Load dataset
df = pd.read_csv("/content/Dataset.csv")

# Preview data
print(df.head())

In [ ]:
print(f"Dataset shape: {df.shape}")

# Check sentiment distribution
print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

In [ ]:
print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

In [ ]:
# Clean text function
def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'<[^>]+>', ' ', text)             # Remove HTML
    text = re.sub(r'http\S+|www\S+', ' ', text)      # Remove URLs
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)         # Remove non-alphabets
    text = re.sub(r'\s+', ' ', text).strip()         # Extra spaces
    return text.lower()

# Apply cleaning
df['cleaned_review'] = df['review'].apply(clean_text)
print(df['cleaned_review'].iloc[0][:300])


In [ ]:
ps = PorterStemmer()
stop_words = set(ENGLISH_STOP_WORDS)

def tokenize_and_stem(text):
    tokens = re.findall(r'\b[a-zA-Z]+\b', text)
    return [ps.stem(w) for w in tokens if w not in stop_words and len(w) > 1]

df['tokenized_review'] = df['cleaned_review'].apply(tokenize_and_stem)
print(df['tokenized_review'].iloc[0][:15])


In [ ]:
df = df.dropna(subset=['review', 'sentiment'])
df = df.drop_duplicates(subset=['review', 'sentiment'])
print(f"Dataset shape after cleaning: {df.shape}")
print(df['sentiment'].value_counts())


In [ ]:
# TF-IDF Vectorizer for Logistic Regression
tfidf = TfidfVectorizer(ngram_range=(1,3), max_features=5000, stop_words="english")
X_tfidf = tfidf.fit_transform(df['cleaned_review'])
print("TF-IDF shape:", X_tfidf.shape)


In [ ]:
# LSTM Tokenizer and Padding
MAX_WORDS = 5000
MAX_LEN = 200  # adjust if needed based on review lengths

# Tokenizer on cleaned text (no stemming)
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_review'])

# Convert text to sequences and pad
sequences = tokenizer.texts_to_sequences(df['cleaned_review'])
X_padded = pad_sequences(sequences, maxlen=MAX_LEN, padding="post")

print("Padded sequences shape:", X_padded.shape)


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Encode labels (negative=0, positive=1)
le = LabelEncoder()
y = le.fit_transform(df['sentiment'])
classes = le.classes_
print("Classes:", classes)

# Split for Logistic Regression (TF-IDF)
X_tfidf_train, X_tfidf_test, y_tfidf_train, y_tfidf_test = train_test_split(
    X_tfidf, y, test_size=0.3, stratify=y, random_state=SEED
)

# Split for LSTM (padded sequences)
X_lstm_train, X_lstm_test, y_lstm_train, y_lstm_test = train_test_split(
    X_padded, y, test_size=0.3, stratify=y, random_state=SEED
)



In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

logreg = LogisticRegression(max_iter=1000, random_state=SEED)
logreg.fit(X_tfidf_train, y_tfidf_train)

y_pred_lr = logreg.predict(X_tfidf_test)
print("Logistic Regression Results")
print("Accuracy:", accuracy_score(y_tfidf_test, y_pred_lr))
print(classification_report(y_tfidf_test, y_pred_lr, target_names=classes))
print("Confusion Matrix:\n", confusion_matrix(y_tfidf_test, y_pred_lr))


In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Bidirectional, Dropout, Dense

EMBEDDING_DIM = 100

# Define a slightly deeper and smaller LSTM model
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=EMBEDDING_DIM),
    Bidirectional(LSTM(64, return_sequences=True)),
    Dropout(0.5),
    LSTM(32),
    Dropout(0.3),
    Dense(1, activation='sigmoid')  # binary classification
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Build explicitly so summary works
model.build(input_shape=(None, MAX_LEN))
model.summary()


In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Train LSTM
history = model.fit(
    X_lstm_train, y_lstm_train,
    validation_split=0.1,
    epochs=20,
    batch_size=64,
    callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
    verbose=1
)

# Predictions
y_pred_probs = model.predict(X_lstm_test)
y_pred_lstm = (y_pred_probs.flatten() > 0.5).astype(int)  # threshold for binary

# Evaluation
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("LSTM Results")
print("Accuracy:", accuracy_score(y_lstm_test, y_pred_lstm))
print(classification_report(y_lstm_test, y_pred_lstm, target_names=classes))
print("Confusion Matrix:\n", confusion_matrix(y_lstm_test, y_pred_lstm))


In [ ]:
import matplotlib.pyplot as plt

# Plot accuracy
plt.plot(history.history['accuracy'], label='Train Acc')
plt.plot(history.history['val_accuracy'], label='Val Acc')
plt.title("LSTM Accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

# Plot loss
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title("LSTM Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Confusion matrices
cm_lr = confusion_matrix(y_tfidf_test, y_pred_lr)
cm_lstm = confusion_matrix(y_lstm_test, y_pred_lstm)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Logistic Regression Confusion Matrix
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=axes[0])
axes[0].set_title("Confusion Matrix - Logistic Regression")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# LSTM Confusion Matrix
sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Greens',
            xticklabels=classes, yticklabels=classes, ax=axes[1])
axes[1].set_title("Confusion Matrix - LSTM")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

# Metrics for TF-IDF + Logistic Regression
acc_tfidf = accuracy_score(y_tfidf_test, y_pred_lr)
f1_tfidf = f1_score(y_tfidf_test, y_pred_lr, average="weighted")
prec_tfidf = precision_score(y_tfidf_test, y_pred_lr, average="weighted")
rec_tfidf = recall_score(y_tfidf_test, y_pred_lr, average="weighted")

# Metrics for LSTM
acc_lstm = accuracy_score(y_lstm_test, y_pred_lstm)
f1_lstm = f1_score(y_lstm_test, y_pred_lstm, average="weighted")
prec_lstm = precision_score(y_lstm_test, y_pred_lstm, average="weighted")
rec_lstm = recall_score(y_lstm_test, y_pred_lstm, average="weighted")

# Combine results
comparison = pd.DataFrame({
    "Model": ["Logistic Regression (TF-IDF)", "LSTM"],
    "Accuracy": [acc_tfidf, acc_lstm],
    "F1 Score": [f1_tfidf, f1_lstm],
    "Precision": [prec_tfidf, prec_lstm],
    "Recall": [rec_tfidf, rec_lstm]
})

print(comparison)

# Plot comparison
comparison.set_index("Model").plot(kind="bar", figsize=(8,5))
plt.title("Model Performance Comparison")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.ylim(0,1)
plt.legend(loc="lower right")
plt.show()


In [ ]:
import joblib

# -----------------------------
# 1. Save Logistic Regression + TF-IDF
# -----------------------------
joblib.dump(logreg, "tfidf_logreg_model.pkl")
joblib.dump(tfidf, "tfidf_vectorizer.pkl")
print("Saved: tfidf_logreg_model.pkl, tfidf_vectorizer.pkl")

# -----------------------------
# 2. Save LSTM tokenizer & training history
# -----------------------------
joblib.dump(tokenizer, "lstm_tokenizer.pkl")
joblib.dump(history.history, "lstm_history.pkl")  # training history
print("Saved: lstm_tokenizer.pkl, lstm_history.pkl")

# -----------------------------
# 3. Save LSTM model separately
# -----------------------------
model.save("lstm_model.h5")
print("Saved: lstm_model.h5")

print("All models and artifacts have been saved successfully!")
